# 33 — AWS Security: IAM, KMS & Secrets Manager

**Role:** Senior AWS DevOps Engineer | **Focus:** Security Best Practices, IAM, KMS, Secrets Management

---

## Section A: IAM Deep Dive

### Q1: Least Privilege Implementation
**Question:** How do you implement least privilege for a development team that needs to debug production EKS pods but should NOT be able to modify infrastructure?

**Expected Answer:**
- **IAM Policy**: Allow `eks:DescribeCluster`, deny `eks:DeleteCluster`, `eks:UpdateClusterConfig`.
- **K8s RBAC**: ClusterRole with `get`, `list`, `logs` on pods/services. No `create`, `delete`, `exec`.
- **aws-auth ConfigMap**: Map their IAM role to the K8s RBAC group.
- **SSM Session Manager**: Allow `ssm:StartSession` to nodes instead of SSH. Logged and auditable.
- **Conditions**: Restrict by source IP, MFA requirement, or time window.

---

### Q2: IAM Roles for Service Accounts (IRSA)
**Question:** What is IRSA and why is it better than attaching an IAM role to an EC2 node?

**Expected Answer:**
- **Node-level role**: ALL pods on the node get the same permissions. Over-privileged.
- **IRSA**: Each pod gets its own IAM role via a K8s ServiceAccount. Uses OIDC federation.
- **How it works**: EKS OIDC provider → IAM trust policy trusts `system:serviceaccount:namespace:sa-name` → Pod gets temporary STS credentials.
- **Example**: Only the `payment-service` pod can access the `payments` S3 bucket.
- **Alternative**: EKS Pod Identity (newer, simpler setup but same concept).

---

### Q3: IAM Policy Boundaries
**Question:** What are IAM Permission Boundaries and when would you use them?

**Expected Answer:**
- **Permission Boundary**: A managed policy that sets the MAXIMUM permissions a role/user can have, regardless of their attached policies.
- **Use case**: Allowing developers to create IAM roles (for Lambda, ECS tasks) without letting them escalate privileges beyond a defined boundary.
- **Example**: Dev creates a Lambda execution role. Boundary ensures they can't add `iam:*` or `s3:DeleteBucket`.
- **Combined with SCPs**: SCPs at the Organization level + Boundaries at the account level = defense in depth.

## Section B: Encryption & KMS

### Q4: KMS Key Management
**Question:** How do you use AWS KMS to encrypt data at rest across RDS, S3, and EKS secrets?

**Expected Answer:**
- **CMK (Customer Managed Keys)**: Full control over rotation, policies, deletion schedule. Use for production data.
- **RDS**: Enable encryption at creation. Cannot encrypt an existing unencrypted DB (must snapshot → copy encrypted → restore).
- **S3**: Default encryption with SSE-KMS. Bucket policy can deny `PutObject` without encryption header.
- **EKS Secrets**: Envelope encryption. etcd stores secrets encrypted with a KMS-managed DEK.
- **Key policy**: Separate key administrators (manage key) from key users (encrypt/decrypt).
- **Rotation**: Enable automatic annual rotation. Old key versions retained for decryption.

---

### Q5: Secrets Manager vs. Parameter Store
**Question:** When do you use AWS Secrets Manager vs. SSM Parameter Store?

**Expected Answer:**

| Feature | Secrets Manager | SSM Parameter Store |
|---|---|---|
| Auto-rotation | Yes (Lambda-based) | No |
| Cross-account | Yes | Limited |
| Cost | $0.40/secret/month | Free (standard) |
| Best for | DB creds, API keys | Config values, feature flags |
| K8s integration | External Secrets Operator | External Secrets Operator |

- **K8s integration**: Use **External Secrets Operator** to sync AWS secrets → K8s Secrets automatically.
- **RDS rotation**: Secrets Manager has built-in RDS password rotation with a Lambda function.

---

### Q6: Secrets in Kubernetes
**Question:** K8s Secrets are base64-encoded, not encrypted. How do you secure them in production?

**Expected Answer:**
- **EKS Envelope Encryption**: Enable KMS encryption for etcd secrets.
- **External Secrets Operator**: Secrets stored in AWS Secrets Manager; synced to K8s at runtime.
- **Sealed Secrets**: Encrypt secrets client-side; only the cluster can decrypt (Bitnami Sealed Secrets).
- **RBAC**: Restrict `get` on secrets to only the service accounts that need them.
- **Audit**: Enable K8s audit logging to track who accessed which secrets.

## Section C: Network Security & Compliance

### Q7: WAF & DDoS Protection
**Question:** How do you protect a public-facing API on EKS from common web attacks and DDoS?

**Expected Answer:**
- **AWS WAF**: Attach to ALB or API Gateway. Rules for SQL injection, XSS, rate limiting.
- **AWS Shield Standard**: Free DDoS protection at L3/L4 (included with ALB).
- **Shield Advanced**: $3K/month. DDoS response team, cost protection.
- **CloudFront**: CDN in front absorbs volumetric attacks. Origin is the ALB.
- **Rate limiting**: WAF rate-based rules (e.g., 2000 requests per 5 min per IP).
- **Bot control**: AWS WAF Bot Control or managed rule groups.

---

### Q8: Security Scanning & Compliance
**Question:** How do you automate security scanning for container images and infrastructure?

**Expected Answer:**
- **Container images**: ECR image scanning (on push), Trivy in CI pipeline, Snyk for dependency scanning.
- **Infrastructure**: Checkov/tfsec for Terraform, AWS Config Rules for runtime compliance.
- **Runtime**: Falco for runtime threat detection in K8s (unexpected process execution, network connections).
- **GuardDuty**: Enable for EKS audit log monitoring, EC2 threat detection.
- **Security Hub**: Aggregates findings from GuardDuty, Inspector, Config. Single pane of glass.
- **CIS Benchmarks**: `kube-bench` to validate EKS nodes against CIS Kubernetes Benchmark.